In [4]:
import json
from pathlib import Path

SEMESTER = "fal26"

# ======== CONFIG ========
INPUT_JSON = f"{SEMESTER}.json"     
OUTPUT_SQL = f"{SEMESTER}.sql"      
INCLUDE_TRANSACTION = True      # USE course_catalog

# ======== HELPERS ========
def sql_escape(value):
    if value is None:
        return "NULL"
    if isinstance(value, str):
        return "'" + value.replace("'", "''") + "'"
    return str(value)

def generate_sql(data, upsert=False):
    course_inserts = []
    meeting_inserts = []

    for course in data:
        crn = course.get("crn")

        # --- Course ---
        course_sql = f"""
INSERT INTO course (
    crn,
    course_code,
    department,
    course_title,
    grad_requirements,
    enrollment_current,
    enrollment_max,
    enrollment_remaining
) VALUES (
    {crn},
    {sql_escape(course.get("course_code"))},
    {sql_escape(course.get("department"))},
    {sql_escape(course.get("course_title"))},
    {sql_escape(course.get("grad_requirements"))},
    {course.get("enrollment", {}).get("current", 0)},
    {course.get("enrollment", {}).get("max", 0)},
    {course.get("enrollment", {}).get("remaining", 0)}
)"""

        course_sql += ";"
        course_inserts.append(course_sql.strip())

        # --- Meetings ---
        for m in course.get("meetings", []):
            meeting_sql = f"""
INSERT INTO meeting (
    crn,
    meeting_type,
    weekdays,
    start_time,
    end_time,
    class_time,
    room
) VALUES (
    {crn},
    {sql_escape(m.get("meeting_type"))},
    {sql_escape(m.get("weekdays"))},
    {sql_escape(m.get("start_time"))},
    {sql_escape(m.get("end_time"))},
    {sql_escape(m.get("class_time"))},
    {sql_escape(m.get("room"))}
);
""".strip()

            meeting_inserts.append(meeting_sql)

    return course_inserts + meeting_inserts


# ======== MAIN EXECUTION ========
with open(INPUT_JSON, "r") as f:
    data = json.load(f)

seen = set()
unique_data = []

for course in data:
    crn = course.get("crn")
    if crn not in seen:
        seen.add(crn)
        unique_data.append(course)

data = unique_data

statements = generate_sql(data, upsert=UPSERT)

# Combine into one runnable script
full_sql = "\n\n".join(statements)

if INCLUDE_TRANSACTION:
    full_sql = f"USE {SEMESTER};\n\n{full_sql}"

# Output
if OUTPUT_SQL:
    Path(OUTPUT_SQL).write_text(full_sql)
    print(f"SQL written to {OUTPUT_SQL}")
else:
    print(full_sql[:2000])  # avoid flooding notebook

SQL written to sql_code.sql
